In [1]:
# --- STEP 0: System Setup ---
! git clone https://github.com/aqwertyuiop48/edu-101-java-code.git
print("🔧 Installing Java 17, Temporal CLI, and Maven...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install openjdk-17-jdk-headless maven -y > /dev/null
!ls

Cloning into 'edu-101-java-code'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 159 (delta 45), reused 151 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 30.07 MiB | 5.18 MiB/s, done.
Resolving deltas: 100% (45/45), done.
🔧 Installing Java 17, Temporal CLI, and Maven...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 33.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-

In [2]:
# 1. Download using the correct 2026 URL
!curl -sSf https://temporal.download/cli.sh | sh

# 2. Add the binary to the Python environment's PATH
import os
temporal_bin_path = os.path.expanduser("~/.temporalio/bin")
os.environ['PATH'] += f":{temporal_bin_path}"

# 3. Create a system-wide link so all '!temporal' commands work seamlessly
!ln -sf /root/.temporalio/bin/temporal /usr/local/bin/temporal

# 4. Verify it works!
print("\n✅ Verification:")
!temporal --version

%cd /content/edu-101-java-code/example1

temporal: Downloading Temporal CLI latest
temporal: Temporal CLI installed at /root/.temporalio/bin/temporal
temporal: For convenience, we recommend adding it to your PATH
temporal: If using bash, run echo export PATH="\$PATH:/root/.temporalio/bin" >> ~/.bashrc

✅ Verification:
temporal version 1.6.1 (Server 1.30.1, UI 2.45.3)
/content/edu-101-java-code/example1


## --- EXAMPLE 1 ---

In [3]:
import subprocess
import time

# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 1...")
# Make sure we are in the right folder!
%cd /content/edu-101-java-code/example1

print("🛰️ Terminal 1: Starting Temporal Server (Port 8081)...")
# Popen starts the server in the background and moves to the next line immediately
import subprocess
import time

# 1. Start the Temporal server in the background (equivalent to trailing '&')
print("Starting Temporal server...")
server_process = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8081','--db-filename','cluster2.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")

# Note: The server_process is still running in the background here.
# You can now run your Gradle command via subprocess, or keep the script running.

print("👷 Terminal 2: Compiling...")
!mvn clean compile > /dev/null

print("👷 Terminal 2: Starting Java Worker...")
# Run the worker in the background too
worker_process = subprocess.Popen(['mvn', 'exec:java', '-Dexec.mainClass=helloworkflow.SayHelloWorker'])

print("⏳ Waiting 15 seconds for server and worker to start...")
time.sleep(15)

print("🎬 Terminal 3: Running Starter...")
# We use ! for the starter because we DO want the notebook to wait for this one to finish
!mvn exec:java -Dexec.mainClass="helloworkflow.Starter"

^C

🏗️ Setting up Example 1...
/content/edu-101-java-code/example1
🛰️ Terminal 1: Starting Temporal Server (Port 8081)...
Starting Temporal server...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
👷 Terminal 2: Compiling...
👷 Terminal 2: Starting Java Worker...
⏳ Waiting 15 seconds for server and worker to start...
🎬 Terminal 3: Running Starter...
[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.example:temporal-demo >----------------------
[INFO] Building temporal-demo 1.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- exec-maven-plugin:3.6.3:java (default-cli) @ temporal-demo ---
[helloworkflow.Starter.main()] INFO io.temporal.serviceclient.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}}
Workflow result: Hello Tem

## --- EXAMPLE 2 ---


In [4]:
import subprocess
import time

# Cleanup everything first
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 2...")
# Move to root, clone, then move inside
%cd /content
# Only clone if the folder doesn't exist yet
![ -d "samples-java" ] || git clone https://github.com/temporalio/samples-java
%cd samples-java

print("🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...")
# Popen runs the server in the background without blocking the cell
# Note: I removed '--port 7235' so it defaults to 7233. This allows the sample code to connect!
server_process_2 = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8082','--db-filename','cluster3.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")
print("⏳ Waiting 10 seconds for server to boot...")
time.sleep(10)

print("👷 Terminal 5: Building and Executing HelloAccumulator...")
# We use ! here because we WANT the cell to wait for Gradle to finish and print the result
!./gradlew build > /dev/null
#!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

^C

🏗️ Setting up Example 2...
/content
Cloning into 'samples-java'...
remote: Enumerating objects: 7772, done.
remote: Counting objects: 100% (2449/2449), done.
remote: Compressing objects: 100% (776/776), done.
remote: Total 7772 (delta 1920), reused 1735 (delta 1635), pack-reused 5323 (from 1)
Receiving objects: 100% (7772/7772), 3.53 MiB | 26.61 MiB/s, done.
Resolving deltas: 100% (3840/3840), done.
/content/samples-java
🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
⏳ Waiting 10 seconds for server to boot...
👷 Terminal 5: Building and Executing HelloAccumulator...
Note: Some input files use or override a deprecated API.
Note: Recompile with -Xlint:deprecation for details.
Note: Some input files use unchecked or unsafe operations.
Note: Recompile with -Xlint:unchecked for details.
/content/samples-java/core/src/test/java/io/temporal/samples/hello/H

In [5]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

13:18:42.232 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:18:42.904 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=7654@a7daa50f1719} 
13:18:42.929 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=7654@a7daa50f1719} 
Worker started for task queue: HelloAccumulatorTaskQueue
13:18:43.549 {HelloAccumulatorWorkflow-green } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - received greeting-Greeting [greetingText=Abe Robot, bucket=green, greetingKey=key-0] 
13:18:43.549 {HelloAccumulatorWorkflow-red } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - received

In [6]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivity


13:20:50.148 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:20:50.823 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=8290@a7daa50f1719} 
13:20:50.855 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=8290@a7daa50f1719} 
13:20:51.423 {HelloActivityWorkflow 76fcbf7c-a5ea-3d5b-85e4-23309671bb19} [Activity Executor taskQueue="HelloActivityTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
Hello World!


In [7]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityRetry


13:20:57.433 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:20:58.600 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=8408@a7daa50f1719} 
13:20:58.668 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=8408@a7daa50f1719} 
composeGreeting activity is going to fail
13:20:59.600 {HelloActivityWithRetriesWorkflow 6d8c8ca4-0da5-3e60-acec-5a4d30b6462e} [Activity Executor taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=6d8c8ca4-0da5-3e60-acec-5a4d30b6462e, activityType=ComposeGreeting, attempt=1 
j

In [8]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityExclusiveChoice


13:21:11.427 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:21:12.746 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=8563@a7daa50f1719} 
13:21:12.790 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=8563@a7daa50f1719} 
Order results: Ordered 4 Oranges...Ordered 5 Bananas...Ordered 8 Apples...Ordered 1 Cherries...


In [9]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsync


13:21:19.622 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:21:20.213 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=8696@a7daa50f1719} 
13:21:20.237 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=8696@a7daa50f1719} 
Hello World!
Bye World!


In [10]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloParallelActivity


13:21:25.383 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:21:26.032 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=8812@a7daa50f1719} 
13:21:26.063 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=8812@a7daa50f1719} 
Hello John!
Hello Mary!
Hello Michael!
Hello Jennet!


In [11]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncActivityCompletion


13:21:33.737 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:21:34.734 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=8942@a7daa50f1719} 
13:21:34.769 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=8942@a7daa50f1719} 
Hello World!


In [12]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncLambda


13:21:39.630 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:21:40.297 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=9064@a7daa50f1719} 
13:21:40.325 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=9064@a7daa50f1719} 
Hello World!
Hello World!


In [13]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDetachedCancellationScope


13:21:46.054 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:21:47.246 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=9186@a7daa50f1719} 
13:21:47.295 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=9186@a7daa50f1719} 
13:21:50.129 {HelloDetachedCancellationWorkflow 166f2305-c32b-3624-8eed-6d02c3f2f94c} [Activity Executor taskQueue="HelloDetachedCancellationTaskQueue", namespace="default": 1] INFO  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity canceled. ActivityId=166f2305-c32b-3624-8eed-6d02c3f2f94c, activityType=SayHello, attempt=1 
Goodbye John!


In [14]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloChild


13:21:54.721 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:21:55.401 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloChildTaskQueue", namespace="default", identity=9321@a7daa50f1719} 
Hello World!


In [15]:
!timeout 240s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloCron
# taking too long

13:22:00.323 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:22:01.033 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=9435@a7daa50f1719} 
13:22:01.077 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=9435@a7daa50f1719} 
Started workflow_id: "HelloCronWorkflow"
run_id: "019cd7e9-5dc7-7ee7-bf72-1f3c60853ff5"

From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!


In [16]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDynamic


13:26:00.956 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:26:01.588 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=10522@a7daa50f1719} 
13:26:01.632 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=10522@a7daa50f1719} 
DynamicACT: Hello John from: DynamicWF


In [17]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloEagerWorkflowStart


13:26:08.021 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:26:09.346 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=10637@a7daa50f1719} 
13:26:09.404 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=10637@a7daa50f1719} 
13:26:10.384 {HelloEagerWorkflowStartWorkflow 7e2a4c13-8d12-3e19-be3b-9dea5f525f1c} [Activity Executor taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloEagerWorkflowStart$GreetingLocalActivitiesImpl - Composing greeting... 
Hello World!


In [18]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloException


13:26:14.759 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:26:15.376 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=10768@a7daa50f1719} 
13:26:15.408 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=10768@a7daa50f1719} 
13:26:16.292 {3457ee4e-db45-34a7-bdc9-36c486b1464b 414a9c82-8c37-3451-8522-01a7a8b4162e} [Activity Executor taskQueue="HelloExceptionTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=414a9c82-8c37-3451-8522-01a7a8b4162e, activityType=ComposeGreeting, attempt=1 
java.io.IOException: Hello World!
	at io.temporal.samples.hello.Hel

In [19]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloLocalActivity


13:26:22.686 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:26:23.841 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloLocalActivity", namespace="default", identity=10891@a7daa50f1719} 
13:26:23.894 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloLocalActivity", namespace="default", identity=10891@a7daa50f1719} 
Hello World!


In [20]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPeriodic


13:26:30.590 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:26:31.284 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=11022@a7daa50f1719} 
13:26:31.322 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=11022@a7daa50f1719} 
Started a new GreetingWorkflow.
Observing the workflow execution for 20 seconds.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5760 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4411 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5958 milliseconds and then I will greet you agai

In [21]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPolymorphicActivity


13:26:58.148 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:26:59.389 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=11224@a7daa50f1719} 
13:26:59.432 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=11224@a7daa50f1719} 
Hello World!
Bye World!



In [22]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloQuery


13:27:04.939 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:27:05.596 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloQueryTaskQueue", namespace="default", identity=11350@a7daa50f1719} 
Hello World!
Bye World!


In [23]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSaga


13:27:15.520 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:27:16.665 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=11475@a7daa50f1719} 
13:27:16.699 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=11475@a7daa50f1719} 
ActivityOperationImpl.execute() is called with amount 10
ActivityOperationImpl.execute() is called with amount 20
Other compensation logic in main workflow.
ActivityCompensationImpl.compensate() is called with amount -20
ActivityCompensationImpl.compensate() is called with amount -10


In [24]:
!timeout 300s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSchedules

13:27:23.148 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:27:23.783 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=11612@a7daa50f1719} 
13:27:23.818 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=11612@a7daa50f1719} 
From HelloScheduleWorkflow-2026-03-10T13:27:24Z: Hello World from HelloSchedule scheduled at 2026-03-10T13:27:24Z!
From HelloScheduleWorkflow-2026-03-10T13:27:25Z: Hello World from HelloSchedule scheduled at 2026-03-10T13:27:25Z!
From HelloScheduleWorkflow-2026-03-10T13:27:30Z: Hello World from HelloSchedule scheduled at 2026-03-10T13:27:30Z!
From HelloScheduleWorkflow-2026-03-10T13:27:35Z: Hello World

In [25]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignal


13:28:18.944 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:28:19.637 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalTaskQueue", namespace="default", identity=11933@a7daa50f1719} 
[Hello World!, Hello Universe!]


In [26]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSearchAttributes

13:28:24.341 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:28:24.977 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=12043@a7daa50f1719} 
13:28:25.001 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=12043@a7daa50f1719} 
In workflow we get CustomKeywordField is: keys
Hello SearchAttributes!


In [27]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloTypedSearchAttributes


13:28:32.051 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:28:32.927 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=12166@a7daa50f1719} 
13:28:32.953 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=12166@a7daa50f1719} 
Hello TypedSearchAttributes how are you doing?


In [28]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSideEffect



13:28:37.647 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:28:38.340 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=12288@a7daa50f1719} 
13:28:38.374 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=12288@a7daa50f1719} 
Goodbye World!
Goodbye World!


In [29]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloUpdate


13:28:43.899 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:28:45.128 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=12403@a7daa50f1719} 
13:28:45.179 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=12403@a7daa50f1719} 
13:28:46.387 {HelloUpdateWorkflow 02e44cd9-d584-3807-a3e5-d106a9d47f4a} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
13:28:46.578 {HelloUpdateWorkflow f0b1ba53-154c-3650-9185-b57d253d55cf} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$Greetin

In [30]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithTimer


13:28:52.631 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:28:53.328 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=12538@a7daa50f1719} 
13:28:53.363 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=12538@a7daa50f1719} 
Processing value: Value 2
Processing value: Value 5
Processing value: Value 7
Processing value: Value 10
13:29:18.939 {HelloSignalWithTimerWorkflow } [workflow-method-HelloSignalWithTimerWorkflow-04e3fd15-a7a0-426b-baed-b1740693b56c] INFO  i.t.s.h.HelloSignalWithTimer$SignalWithTimerWorkflowImpl - Timer canceled via exit signal 
Processing value: Value 11


In [31]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithStartAndWorkflowInit

13:29:23.289 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
13:29:23.944 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=12760@a7daa50f1719} 
13:29:23.975 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=12760@a7daa50f1719} 
Result: Hello Michael Jordan,Hello John Stockton
13:29:24.867 {without-init } [signal addGreeting] WARN  i.t.i.sync.WorkflowExecutionHandler - Workflow execution failure WorkflowId='without-init', RunId=a1aba087-14a6-4375-a121-be8cf94ee24f, WorkflowType='MyWorkflowNoInit' 
java.lang.NullPointerException: Cannot invoke "java.util.List.add(Object)" because "this.peopleToGreet" is null
	at io.temporal.sam

In [32]:
# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

^C
